# [시장분석] 강남3구 아파트, 어느 동이 가장 많이 올랐을까? - 최근 10년 상승률 전수조사

KB 매매 일반가를 이용해 강남구·서초구·송파구의 법정동별 아파트 상승률을 비교합니다.

- 비교기간: 2016년 7월~2026년 7월
- 동일 단지·동일 면적 타입 매칭
- 복수 타입은 단지별 중앙값, 동별 결과는 단지 중앙값
- 100세대 미만·주상복합 제외, 유효 단지 3개 이상인 동만 본 분석에 포함
- `새로 수집` 기본값은 OFF이며 저장 CSV를 사용합니다.


In [ ]:
# @title 강남3구 동별 10년 상승률을 계산하세요
"""강남·서초·송파 법정동별 아파트의 최근 10년 상승률을 계산한다.

KB부동산 매매 일반가를 사용해 2016년 7월과 2026년 7월에 모두 가격이
존재하는 동일 단지·동일 면적 타입을 매칭한다. 타입별 CAGR의 중앙값을
단지 대표값으로, 단지별 CAGR의 중앙값을 동 대표값으로 사용한다.
주상복합은 제외하며 유효 단지가 3개 이상인 동만 본 분석에 포함한다.
"""

from __future__ import annotations

import math
import os
from html import escape
import statistics
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Callable

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp/matplotlib-va-gangnam3")))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, HTML, Image, Markdown, display
from matplotlib import font_manager
from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea


새로_수집 = False  # @param {type:"boolean"}

REFRESH_KB_DATA = bool(새로_수집)
START_YEAR_MONTH = "201607"
END_YEAR_MONTH = "202607"
COMPARISON_YEARS = 10
MIN_COMPLEX_COUNT = 3
MIN_HOUSEHOLDS = 100
MAX_WORKERS = 8
API_COMPLEX = "https://api.kbland.kr/land-complex"
API_PRICE = "https://api.kbland.kr/land-price"

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
COMPLEX_PATH = OUTPUT_DIR / "kb_gangnam3_apartment_complexes.csv"
MATCHED_TYPE_PATH = OUTPUT_DIR / "kb_gangnam3_matched_types_201607_202607.csv"
DONG_SUMMARY_PATH = OUTPUT_DIR / "gangnam3_dong_10y_growth.csv"
DONG_COVERAGE_PATH = OUTPUT_DIR / "gangnam3_dong_10y_coverage.csv"
DONG_CHART_PATH = OUTPUT_DIR / "gangnam3_dong_10y_growth.png"

LEGAL_DONGS = {
    "강남구": {
        "역삼동": "1168010100", "개포동": "1168010300",
        "청담동": "1168010400", "삼성동": "1168010500",
        "대치동": "1168010600", "신사동": "1168010700",
        "논현동": "1168010800", "압구정동": "1168011000",
        "세곡동": "1168011100", "자곡동": "1168011200",
        "율현동": "1168011300", "일원동": "1168011400",
        "수서동": "1168011500", "도곡동": "1168011800",
    },
    "서초구": {
        "방배동": "1165010100", "양재동": "1165010200",
        "우면동": "1165010300", "원지동": "1165010400",
        "잠원동": "1165010600", "반포동": "1165010700",
        "서초동": "1165010800", "내곡동": "1165010900",
        "염곡동": "1165011000", "신원동": "1165011100",
    },
    "송파구": {
        "잠실동": "1171010100", "신천동": "1171010200",
        "풍납동": "1171010300", "송파동": "1171010400",
        "석촌동": "1171010500", "삼전동": "1171010600",
        "가락동": "1171010700", "문정동": "1171010800",
        "장지동": "1171010900", "방이동": "1171011100",
        "오금동": "1171011200", "거여동": "1171011300",
        "마천동": "1171011400",
    },
}

_THREAD_LOCAL = threading.local()


def build_session():
    """KB 공개 API 요청용 세션을 만든다."""
    import requests

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "ko-KR,ko;q=0.9",
        "Origin": "https://kbland.kr",
        "Referer": "https://kbland.kr/",
    })
    return session


def get_worker_session():
    """작업 스레드마다 재사용할 요청 세션을 반환한다."""
    if not hasattr(_THREAD_LOCAL, "session"):
        _THREAD_LOCAL.session = build_session()
    return _THREAD_LOCAL.session


def request_data(base_url: str, endpoint: str, params: dict[str, Any]) -> Any:
    """KB API를 재시도하고 실제 데이터 영역을 반환한다."""
    last_error = None
    for attempt in range(5):
        try:
            response = get_worker_session().get(
                f"{base_url}{endpoint}", params=params, timeout=30
            )
            response.raise_for_status()
            body = response.json().get("dataBody", {})
            if body.get("resultCode") == 33210:
                return []
            return body.get("data", [])
        except Exception as error:
            last_error = error
            time.sleep(0.5 * (attempt + 1))
    raise RuntimeError(f"KB API 요청 실패: {endpoint}, {params}") from last_error


def collect_complexes() -> pd.DataFrame:
    """세 자치구의 법정동별 순수 아파트 단지 목록을 수집한다."""
    rows = []
    for district, dongs in LEGAL_DONGS.items():
        for dong, legal_code in dongs.items():
            items = request_data(
                API_COMPLEX, "/complexComm/hscmList", {"법정동코드": legal_code}
            )
            for item in items:
                if item.get("매물종별구분") != "01":
                    continue
                rows.append({
                    "자치구": district,
                    "동": dong,
                    "법정동코드": legal_code,
                    "단지기본일련번호": int(item["단지기본일련번호"]),
                    "아파트": item["단지명"],
                })
    complexes = (
        pd.DataFrame(rows)
        .drop_duplicates("단지기본일련번호")
        .sort_values(["자치구", "동", "아파트"])
        .reset_index(drop=True)
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    complexes.to_csv(COMPLEX_PATH, index=False, encoding="utf-8-sig")
    return complexes


def run_parallel(
    rows: list[Any], worker: Callable[[Any], Any], label: str
) -> tuple[list[Any], list[Any]]:
    """목록 작업을 병렬 실행하고 성공 결과와 실패 입력을 반환한다."""
    results = []
    failures = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(worker, row): row for row in rows}
        for completed, future in enumerate(as_completed(futures), 1):
            try:
                result = future.result()
                if isinstance(result, list):
                    results.extend(result)
                elif result is not None:
                    results.append(result)
            except RuntimeError:
                failures.append(futures[future])
            if completed % 250 == 0 or completed == len(rows):
                print(f"{label}: {completed:,}/{len(rows):,}", flush=True)
    return results, failures


def collect_types(complexes: pd.DataFrame) -> pd.DataFrame:
    """전체 단지의 모든 면적 타입 목록을 수집한다."""
    def collect_one(row_dict: dict[str, Any]) -> list[dict[str, Any]]:
        complex_id = int(row_dict["단지기본일련번호"])
        items = request_data(
            API_COMPLEX, "/complexComm/typeList", {"단지기본일련번호": complex_id}
        )
        result = []
        for item in items:
            area_id = item.get("면적일련번호")
            private_area = item.get("전용면적")
            if not area_id or not private_area:
                continue
            result.append({
                **row_dict,
                "면적일련번호": int(area_id),
                "공급면적_㎡": float(item.get("공급면적") or 0),
                "전용면적_㎡": float(private_area),
                "주택형": item.get("주택형타입내용", ""),
            })
        return result

    inputs = complexes.to_dict("records")
    rows, failures = run_parallel(inputs, collect_one, "단지 타입 조회")
    for row in failures:
        rows.extend(collect_one(row))
    return pd.DataFrame(rows).drop_duplicates(
        ["단지기본일련번호", "면적일련번호"]
    )


def get_price(complex_id: int, area_id: int, year_month: str) -> int | None:
    """면적 타입의 지정 월 KB 매매 일반가를 반환한다."""
    data = request_data(
        API_PRICE,
        "/price/WholQuotList",
        {
            "단지기본일련번호": complex_id,
            "면적일련번호": area_id,
            "기준년": year_month[:4],
        },
    )
    groups = data.get("시세", []) if isinstance(data, dict) else []
    for group in groups:
        for item in group.get("items", []):
            if item.get("기준년월") == year_month:
                price = item.get("매매일반거래가")
                return int(price) if price else None
    return None


def collect_prices(types: pd.DataFrame) -> pd.DataFrame:
    """시작월 가격이 있는 타입에 한해 종료월 가격과 CAGR을 수집한다."""
    type_rows = types.to_dict("records")

    def collect_start(row: dict[str, Any]) -> dict[str, Any] | None:
        start_price = get_price(
            int(row["단지기본일련번호"]),
            int(row["면적일련번호"]),
            START_YEAR_MONTH,
        )
        if start_price is None:
            return None
        return {**row, "시작가격_만원": start_price}

    start_rows, failures = run_parallel(type_rows, collect_start, "2016년 가격 조회")
    for row in failures:
        result = collect_start(row)
        if result is not None:
            start_rows.append(result)
    print(f"2016년 가격 보유 타입: {len(start_rows):,}개", flush=True)

    def collect_end(row: dict[str, Any]) -> dict[str, Any] | None:
        end_price = get_price(
            int(row["단지기본일련번호"]),
            int(row["면적일련번호"]),
            END_YEAR_MONTH,
        )
        if end_price is None:
            return None
        cagr = (end_price / float(row["시작가격_만원"])) ** (
            1 / COMPARISON_YEARS
        ) - 1
        return {**row, "종료가격_만원": end_price, "연평균상승률": cagr}

    matched_rows, failures = run_parallel(start_rows, collect_end, "2026년 가격 조회")
    for row in failures:
        result = collect_end(row)
        if result is not None:
            matched_rows.append(result)
    matched = pd.DataFrame(matched_rows).sort_values(
        ["자치구", "동", "아파트", "전용면적_㎡"]
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    matched.to_csv(MATCHED_TYPE_PATH, index=False, encoding="utf-8-sig")
    return matched


def collect_completion_years(matched_types: pd.DataFrame) -> pd.DataFrame:
    """매칭 단지의 KB 입주년월에서 준공연도를 추가한다."""
    complex_rows = (
        matched_types[["단지기본일련번호", "면적일련번호"]]
        .drop_duplicates("단지기본일련번호")
        .to_dict("records")
    )

    def collect_one(row: dict[str, Any]) -> dict[str, Any]:
        data = request_data(
            API_COMPLEX, "/complex/main",
            {
                "단지기본일련번호": int(row["단지기본일련번호"]),
                "매물종별구분": "01",
                "면적일련번호": int(row["면적일련번호"]),
            },
        )
        move_in = data.get("입주년월") if isinstance(data, dict) else None
        year = int(str(move_in)[:4]) if move_in and str(move_in)[:4].isdigit() else None
        households = data.get("총세대수") if isinstance(data, dict) else None
        return {
            "단지기본일련번호": int(row["단지기본일련번호"]),
            "준공연도": year,
            "세대수": int(households) if households is not None else None,
        }

    details, failures = run_parallel(complex_rows, collect_one, "준공연도 조회")
    for row in failures:
        details.append(collect_one(row))
    enriched = matched_types.drop(
        columns=["준공연도", "세대수"], errors="ignore"
    ).merge(
        pd.DataFrame(details), on="단지기본일련번호", how="left"
    )
    enriched["준공연도"] = enriched["준공연도"].astype("Int64")
    enriched["세대수"] = enriched["세대수"].astype("Int64")
    enriched.to_csv(MATCHED_TYPE_PATH, index=False, encoding="utf-8-sig")
    return enriched


def calculate_dong_results(
    complexes: pd.DataFrame, matched_types: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """타입을 단지와 동 순서로 통합해 상승률과 표본 현황을 계산한다."""
    if matched_types.empty:
        raise ValueError("두 기준월에 모두 가격이 있는 면적 타입이 없습니다.")
    eligible_types = matched_types[
        matched_types["세대수"].ge(MIN_HOUSEHOLDS)
    ].copy()
    if eligible_types.empty:
        raise ValueError("100세대 이상인 유효 단지가 없습니다.")
    complex_results = (
        eligible_types.groupby(
            ["자치구", "동", "단지기본일련번호", "아파트"], as_index=False
        )
        .agg(
            유효타입수=("면적일련번호", "nunique"),
            준공연도=("준공연도", "first"),
            연평균상승률=("연평균상승률", "median"),
        )
    )
    dong_results = (
        complex_results.groupby(["자치구", "동"], as_index=False)
        .agg(
            유효단지수=("단지기본일련번호", "nunique"),
            유효타입수=("유효타입수", "sum"),
            준공연도중앙값=("준공연도", "median"),
            연평균상승률=("연평균상승률", "median"),
        )
    )
    dong_results["준공연도중앙값"] = (
        dong_results["준공연도중앙값"].map(math.ceil).astype(int)
    )
    dong_results["10년누적상승률"] = (
        (1 + dong_results["연평균상승률"]) ** COMPARISON_YEARS - 1
    )

    all_dongs = pd.DataFrame(
        [
            {"자치구": district, "동": dong}
            for district, dongs in LEGAL_DONGS.items()
            for dong in dongs
        ]
    )
    total_counts = (
        complexes.groupby(["자치구", "동"], as_index=False)
        .agg(전체단지수=("단지기본일련번호", "nunique"))
    )
    coverage = (
        all_dongs.merge(total_counts, on=["자치구", "동"], how="left")
        .merge(
            dong_results[["자치구", "동", "유효단지수", "유효타입수"]],
            on=["자치구", "동"],
            how="left",
        )
        .fillna(0)
    )
    for column in ("전체단지수", "유효단지수", "유효타입수"):
        coverage[column] = coverage[column].astype(int)
    coverage["판정"] = coverage["유효단지수"].ge(MIN_COMPLEX_COUNT).map(
        {True: "포함", False: "제외"}
    )
    included = dong_results[dong_results["유효단지수"].ge(MIN_COMPLEX_COUNT)].copy()
    included = included.sort_values(
        ["연평균상승률", "자치구", "동"], ascending=[False, True, True]
    ).reset_index(drop=True)
    overall_median = included["연평균상승률"].median()
    included["전체중앙값대비"] = included["연평균상승률"] - overall_median
    included.insert(0, "순위", included.index + 1)
    return included, coverage.sort_values(["자치구", "동"]).reset_index(drop=True)


def build_analysis_complex_table_html(matched_types: pd.DataFrame) -> str:
    """강남3구 전체 법정동의 분석 대상 단지 목록을 HTML 표로 만든다."""
    valid_complexes = (
        matched_types[matched_types["세대수"].ge(MIN_HOUSEHOLDS)][
            [
                "자치구", "동", "단지기본일련번호", "아파트",
                "준공연도", "세대수",
            ]
        ]
        .drop_duplicates("단지기본일련번호")
        .sort_values(["자치구", "동", "아파트"])
    )
    complex_groups = {
        key: group.to_dict("records")
        for key, group in valid_complexes.groupby(["자치구", "동"])
    }
    all_dongs = sorted(
        (district, dong)
        for district, dongs in LEGAL_DONGS.items()
        for dong in dongs
    )
    rows = []
    for district, dong in all_dongs:
        complexes = complex_groups.get((district, dong), [])
        links = " · ".join(
            f"<a href='https://kbland.kr/se/c/{int(item['단지기본일련번호'])}' "
            f"target='_blank' rel='noopener'>{escape(str(item['아파트']))}</a>"
            for item in complexes
        )
        complex_names = links or "<span class='empty'>없음</span>"
        rows.append(
            "<tr>"
            f"<td class='center'>{district}</td>"
            f"<td class='center'>{dong}</td>"
            f"<td class='center'>{len(complexes)}개</td>"
            f"<td class='complex-list'>{complex_names}</td>"
            "</tr>"
        )
    return f"""
<style>
.dong-complexes {{ max-width:700px; margin:14px 0 34px; font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; }}
.dong-complexes .brand {{ margin:0 0 5px; color:#64748b; font-size:13px; }}
.dong-complexes h2 {{ margin:0 0 4px; color:#0b0b0b; font-size:20px; line-height:1.3; }}
.dong-complexes .subtitle {{ margin:0 0 14px; color:#64748b; font-size:13px; }}
.dong-complexes .table-wrap {{ overflow:hidden; border:1px solid #f0f2f5; border-radius:12px; }}
.dong-complexes table {{ width:100%; table-layout:fixed; border-collapse:separate; border-spacing:0; color:#1e293b; font-size:13px; font-variant-numeric:tabular-nums; }}
.dong-complexes th {{ padding:11px 2px; background:#2b4a75; color:white; text-align:center; font-weight:700; line-height:1.35; word-break:keep-all; }}
.dong-complexes td {{ padding:11px 3px; border-bottom:1px solid #f0f2f5; line-height:1.45; }}
.dong-complexes tbody tr:nth-child(even) td {{ background:#fafbfc; }}
.dong-complexes tbody tr:last-child td {{ border-bottom:0; }}
.dong-complexes .center {{ text-align:center; white-space:nowrap; }}
.dong-complexes .number {{ text-align:right; white-space:nowrap; }}
.dong-complexes .complex-list {{ text-align:left; }}
.dong-complexes .empty {{ color:#94a3b8; }}
.dong-complexes a {{ color:#5b8fc4; text-decoration:underline; }}
.dong-complexes th:nth-child(1), .dong-complexes th:nth-child(2) {{ width:9%; }}
.dong-complexes th:nth-child(3) {{ width:10%; }}
.dong-complexes th:nth-child(4) {{ width:72%; }}
.dong-complexes .footnote {{ margin:9px 0 0; color:#64748b; font-size:13px; line-height:1.5; }}
</style>
<section class='dong-complexes'>
  <p class='brand'>대도시 연구실</p>
  <h2>강남3구 동별 분석 대상 아파트</h2>
  <p class='subtitle'>자치구·동 오름차순 · 100세대 이상</p>
  <div class='table-wrap'><table>
    <thead><tr><th>자치구</th><th>동</th><th>유효 단지</th><th>분석 대상 아파트</th></tr></thead>
    <tbody>{''.join(rows)}</tbody>
  </table></div>
  <p class='footnote'>※ 2016년 7월과 2026년 7월의 동일 면적 KB 일반가가 모두 존재하는 100세대 이상 순수 아파트<br>※ 단지명을 누르면 KB부동산 단지 페이지가 열립니다.</p>
</section>
"""


def select_font_family() -> str:
    """사용 가능한 한글 글꼴을 우선순위에 따라 선택한다."""
    available = {font.name for font in font_manager.fontManager.ttflist}
    for candidate in (
        "Pretendard", "Apple SD Gothic Neo",
        "Noto Sans CJK KR", "Malgun Gothic",
    ):
        if candidate in available:
            return candidate
    return "DejaVu Sans"


def create_dong_growth_chart(results: pd.DataFrame) -> Path:
    """동별 전체 중앙값 대비 상승률을 가로 발산형 막대그래프로 저장한다."""
    plot_data = results.sort_values("연평균상승률", ascending=False).reset_index(drop=True)
    values = plot_data["전체중앙값대비"] * 100
    colors = ["#2F7DD3" if value >= 0 else "#F06432" for value in values]
    district_colors = {
        "강남구": "#2F7DD3",
        "서초구": "#1FAE7A",
        "송파구": "#F2A000",
    }
    font_family = select_font_family()
    plt.rcParams["font.family"] = font_family
    plt.rcParams["axes.unicode_minus"] = False

    fig = plt.figure(figsize=(10, 15), facecolor="#FFFFFF")
    ax = fig.add_axes([0.23, 0.18, 0.73, 0.67])
    y_positions = list(range(len(plot_data)))
    bars = ax.barh(y_positions, values, color=colors, height=0.62)
    ax.invert_yaxis()
    limit = max(abs(values.min()), abs(values.max())) * 1.90
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color="#777777", linewidth=1.2)
    ax.grid(axis="x", color="#DEDCD6", linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#DEDCD6")
    ax.tick_params(axis="x", colors="#777777", labelsize=15, length=0)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([])
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel(
        "전체 중앙값 대비 차이(%p)",
        fontsize=15, color="#777777", labelpad=12,
    )
    ax.text(
        0.01, 1.018, "← 전체 중앙값 미만", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=13, color="#F06432",
    )
    ax.text(
        0.99, 1.018, "전체 중앙값 초과 →", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=13, color="#2F7DD3",
    )

    for y_position, (_, row) in zip(y_positions, plot_data.iterrows()):
        rank = TextArea(
            f"{int(row['순위'])}  ",
            textprops={
                "color": "#777777", "fontsize": 13,
                "fontfamily": font_family, "fontweight": "bold",
            },
        )
        district = TextArea(
            row["자치구"],
            textprops={
                "color": district_colors[row["자치구"]],
                "fontsize": 13, "fontfamily": font_family,
                "fontweight": "bold",
            },
        )
        dong = TextArea(
            f"  {row['동']}",
            textprops={
                "color": "#777777", "fontsize": 14,
                "fontfamily": font_family, "fontweight": "bold",
            },
        )
        packed_label = HPacker(
            children=[rank, district, dong], align="center", pad=0, sep=0
        )
        label_box = AnnotationBbox(
            packed_label, (0, y_position), xycoords=("axes fraction", "data"),
            xybox=(-10, 0), boxcoords="offset points",
            box_alignment=(1, 0.5), frameon=False, pad=0,
        )
        ax.add_artist(label_box)

    offset = limit * 0.025
    for bar, (_, row), value, color in zip(
        bars, plot_data.iterrows(), values, colors
    ):
        label = f"{value:+.2f}%p ({row['연평균상승률']:.2%})"
        ax.text(
            value + (offset if value >= 0 else -offset),
            bar.get_y() + bar.get_height() / 2,
            label, ha="left" if value >= 0 else "right", va="center",
            fontsize=12, fontweight="bold", color=color,
        )

    overall_median = plot_data["연평균상승률"].median()
    fig.text(0.02, 0.972, "대도시 연구실", fontsize=13, color="#64748B")
    fig.text(
        0.02, 0.944, "강남3구 동별 아파트 상승률 | 최근 10년",
        fontsize=21, fontweight="bold", color="#0B0B0B",
    )
    fig.text(
        0.02, 0.914,
        "KB 매매 일반가 · 2016년 7월~2026년 7월 · 100세대 이상 · 전체 중앙값 기준",
        fontsize=16, color="#64748B",
    )
    footnote = (
        f"※ 전체 중앙값은 분석 대상 {len(plot_data)}개 동의 연평균 상승률 중앙값 {overall_median:.2%}\n"
        "※ 막대 라벨: 해당 동 연평균 상승률 (전체 중앙값 대비)\n"
        "※ 동일 단지의 복수 면적은 중앙값으로 통합하고, 동별 단지 상승률의 중앙값 사용\n"
        "※ 100세대 이상이며 유효 단지 3개 이상인 동만 분석 · 주상복합 제외"
    )
    fig.text(
        0.02, 0.11, footnote, fontsize=13, color="#64748B",
        linespacing=1.22, va="top",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(DONG_CHART_PATH, dpi=200, bbox_inches="tight", facecolor="#FFFFFF")
    plt.close(fig)
    return DONG_CHART_PATH


def build_survey_criteria_markdown(
    complexes: pd.DataFrame, matched_types: pd.DataFrame, coverage: pd.DataFrame
) -> str:
    """저장 자료를 기준으로 조사 범위와 표본 조건을 정리한다."""
    complex_count = complexes["단지기본일련번호"].nunique()
    matched_complex_count = matched_types["단지기본일련번호"].nunique()
    matched_type_count = len(matched_types)
    eligible_complex_count = matched_types.loc[
        matched_types["세대수"].ge(MIN_HOUSEHOLDS), "단지기본일련번호"
    ].nunique()
    included_count = int(coverage["판정"].eq("포함").sum())
    excluded_count = int(coverage["판정"].eq("제외").sum())
    excluded_names = "·".join(
        str(dong).removesuffix("동")
        for dong in coverage.loc[coverage["판정"].eq("제외"), "동"]
    )
    return f"""### ■ 데이터 산출 기준

가격 기준: KB부동산 매매 일반가  
조사 대상: 강남구·서초구·송파구 소재 아파트 {complex_count:,}개 전수 조사  
비교 기간: 2016년 7월~2026년 7월 (10년간)  
매칭 규모: 동일 단지·동일 면적 {matched_complex_count:,}개 단지, {matched_type_count:,}개 타입  
유효 표본: 100세대 이상 {eligible_complex_count:,}개 단지 (유효 단지 {MIN_COMPLEX_COUNT}개 이상 {included_count}개 동)  
제외 대상: 100세대 미만, 주상복합, 표본 부족 {excluded_count}개 동({excluded_names})  
산출 방식: 타입 ➔ 단지 ➔ 동 연평균 상승률 전 과정 중앙값(Median) 적용"""


def display_file_link(label: str, path: Path) -> None:
    """Colab에서는 다운로드 버튼을, 로컬에서는 파일 링크를 표시한다."""
    if IS_COLAB:
        import ipywidgets as widgets
        from google.colab import files

        button = widgets.Button(description=f"{label} 다운로드", icon="download")
        button.on_click(lambda _: files.download(str(path)))
        display(button)
    else:
        display(FileLink(str(path), result_html_prefix=f"{label}: "))


def main() -> None:
    """데이터를 수집하거나 캐시를 읽어 동별 10년 상승률을 출력한다."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if REFRESH_KB_DATA:
        print("실행 모드: KB 자료 새로 수집 ON", flush=True)
        complexes = collect_complexes()
        print(f"순수 아파트 단지: {len(complexes):,}개", flush=True)
        types = collect_types(complexes)
        print(f"전체 면적 타입: {len(types):,}개", flush=True)
        matched_types = collect_prices(types)
        matched_types = collect_completion_years(matched_types)
    elif COMPLEX_PATH.exists() and MATCHED_TYPE_PATH.exists():
        print("실행 모드: KB 자료 새로 수집 OFF — 저장 CSV 사용", flush=True)
        complexes = pd.read_csv(COMPLEX_PATH, dtype={"법정동코드": str})
        matched_types = pd.read_csv(MATCHED_TYPE_PATH)
    else:
        display(Markdown(
            "> 저장된 전수조사 결과가 없습니다.  \n"
            "> `새로 수집` 옵션을 체크한 뒤 다시 실행해 주세요."
        ))
        return

    results, coverage = calculate_dong_results(complexes, matched_types)
    results.to_csv(DONG_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    coverage.to_csv(DONG_COVERAGE_PATH, index=False, encoding="utf-8-sig")
    chart_path = create_dong_growth_chart(results)
    display(Markdown(build_survey_criteria_markdown(
        complexes, matched_types, coverage
    )))
    display(HTML(build_analysis_complex_table_html(matched_types)))
    display(Image(filename=str(chart_path)))
    display_file_link("동별 상승률 그래프", chart_path)
    display_file_link("동별 상승률 CSV", DONG_SUMMARY_PATH)
    display_file_link("동별 표본 현황 CSV", DONG_COVERAGE_PATH)
    display_file_link("동일 타입 매칭 상세 CSV", MATCHED_TYPE_PATH)


main()
